In [65]:
list(Block)

[<Block.FLOOR: 0>,
 <Block.WALL: 1>,
 <Block.DIRT: 2>,
 <Block.IRON: 3>,
 <Block.WOOD: 4>,
 <Block.STONE: 5>,
 <Block.SAND: 6>]

In [81]:
from enum import IntEnum
from functools import partial
import numpy as np
import random


class Block(IntEnum):
    FLOOR = 0
    WALL = 1

    DIRT = 2
    IRON = 3
    WOOD = 4
    STONE = 5
    SAND = 6


BREAKABLE_BLOCKS = {
    Block.DIRT,
    Block.IRON,
    Block.WOOD,
    Block.STONE,
    Block.SAND,
}


class Game:
    def __init__(self, size=16, wall_probability=0.03):
        self.size = size

        self.map = np.full(
            (size, size),
            Block.FLOOR,
            dtype=np.int8
        )

        self.player_x = size // 2
        self.player_y = size // 2

        self.selected_block = Block.DIRT

        self.inventory = {
            Block.DIRT: 0,
            Block.IRON: 0,
            Block.WOOD: 0,
            Block.STONE: 0,
            Block.SAND: 0,
        }

        self.actions = {
            # movement
            "move_up": partial(self.move, "up"),
            "move_down": partial(self.move, "down"),
            "move_left": partial(self.move, "left"),
            "move_right": partial(self.move, "right"),

            # break
            "break_up": partial(self.break_block, "up"),
            "break_down": partial(self.break_block, "down"),
            "break_left": partial(self.break_block, "left"),
            "break_right": partial(self.break_block, "right"),

            # place
            "place_up": partial(self.place_block, "up"),
            "place_down": partial(self.place_block, "down"),
            "place_left": partial(self.place_block, "left"),
            "place_right": partial(self.place_block, "right"),

            # inventory selection
            
        } | {f"select_{i}": partial(self.select_block, i) for i in BREAKABLE_BLOCKS}

        self._generate_world(wall_probability)
        

    # ==========================================================
    # Генерация мира
    # ==========================================================

    def _generate_world(self, wall_probability):
        self._generate_border_resources()
        self._generate_walls(wall_probability)

        self.map[self.player_y, self.player_x] = Block.FLOOR
    def _generate_resource_cluster(
        self,
        start_x,
        start_y,
        block_type,
        size
    ):
        x = start_x
        y = start_y

        for _ in range(size):
            if self.inside(x, y):
                self.map[y, x] = block_type

            dx, dy = random.choice([
                (-1, 0),
                (1, 0),
                (0, -1),
                (0, 1),
            ])

            x += dx
            y += dy

            x = max(1, min(self.size - 2, x))
            y = max(1, min(self.size - 2, y))
    def _generate_border_resources(self):
        resources = [
            Block.DIRT,
            Block.IRON,
            Block.WOOD,
            Block.STONE,
            Block.SAND,
        ]

        corners = [
            (2, 2),
            (2, self.size - 3),
            (self.size - 3, 2),
            (self.size - 3, self.size - 3),
        ]

        for corner_x, corner_y in corners:

            for _ in range(5):
                block = random.choice(resources)

                x = corner_x + random.randint(-2, 2)
                y = corner_y + random.randint(-2, 2)

                cluster_size = random.randint(20, 60)

                self._generate_resource_cluster(
                    x,
                    y,
                    block,
                    cluster_size
                )
    def _grow_wall(self, start_x, start_y, length):
        x = start_x
        y = start_y

        dx, dy = random.choice([
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1),
        ])

        for _ in range(length):

            if not self.inside(x, y):
                break

            if (x, y) != (self.player_x, self.player_y):
                self.map[y, x] = Block.WALL

            if random.random() < 0.25:
                dx, dy = random.choice([
                    (-1, 0),
                    (1, 0),
                    (0, -1),
                    (0, 1),
                ])

            x += dx
            y += dy
    def _generate_walls(self, probability):

        wall_chains = int(self.size * probability * 10)

        for _ in range(wall_chains):

            x = random.randint(1, self.size - 2)
            y = random.randint(1, self.size - 2)

            length = random.randint(
                self.size // 3,
                self.size
            )

            self._grow_wall(x, y, length)
    # ==========================================================
    # Вспомогательные методы
    # ==========================================================

    def inside(self, x, y):
        return 0 <= x < self.size and 0 <= y < self.size

    def cell(self, x, y):
        return Block(self.map[y, x])

    def _direction_to_delta(self, direction):
        directions = {
            "up": (0, -1),
            "down": (0, 1),
            "left": (-1, 0),
            "right": (1, 0),
        }
        return directions[direction]

    # ==========================================================
    # Движение
    # ==========================================================

    def move(self, direction):
        dx, dy = self._direction_to_delta(direction)

        nx = self.player_x + dx
        ny = self.player_y + dy

        if not self.inside(nx, ny):
            return False

        block = self.cell(nx, ny)

        if block != Block.FLOOR:
            return False

        self.player_x = nx
        self.player_y = ny

        return True

    # ==========================================================
    # Ломание блоков
    # ==========================================================

    def break_block(self, direction):
        dx, dy = self._direction_to_delta(direction)

        x = self.player_x + dx
        y = self.player_y + dy

        if not self.inside(x, y):
            return False

        block = self.cell(x, y)

        if block not in BREAKABLE_BLOCKS:
            return False

        self.inventory[block] += 1
        self.map[y, x] = Block.FLOOR

        return True

    # ==========================================================
    # Установка блоков
    # ==========================================================

    def place_block(self, direction):
        block = self.selected_block

        if self.inventory[block] <= 0:
            return False

        dx, dy = self._direction_to_delta(direction)

        x = self.player_x + dx
        y = self.player_y + dy

        if not self.inside(x, y):
            return False

        if self.cell(x, y) != Block.FLOOR:
            return False

        self.map[y, x] = block
        self.inventory[block] -= 1

        return True

    # ==========================================================
    # Выбор блока
    # ==========================================================

    def select_block(self, slot):
        mapping = {
            1: Block.DIRT,
            2: Block.IRON,
            3: Block.WOOD,
            4: Block.STONE,
            5: Block.SAND,
        }

        if slot in mapping:
            self.selected_block = mapping[slot]

    # ==========================================================
    # Получение состояния для GUI
    # ==========================================================

    def get_state(self):
        return {
            "map": self.map.copy(),
            "player_x": self.player_x,
            "player_y": self.player_y,
            "inventory": dict(self.inventory),
            "selected": self.selected_block,
        }
    def get_valid_actions(self):
        valid = []

        for name, action in self.actions.items():

            # simulate safely
            try:
                old_map = self.map.copy()
                old_x, old_y = self.player_x, self.player_y
                old_inv = self.inventory.copy()
                old_sel = self.selected_block

                result = action()

                # restore (IMPORTANT)
                self.map = old_map
                self.player_x = old_x
                self.player_y = old_y
                self.inventory = old_inv
                self.selected_block = old_sel

                if result is not False:
                    valid.append(name)

            except:
                continue

        return valid

In [106]:
import tkinter as tk
import time

class GameGUI:
    CELL_SIZE = 24

    COLORS = {
        Block.FLOOR: "#e0e0e0",
        Block.WALL: "#202020",

        Block.DIRT: "#8b5a2b",
        Block.IRON: "#c0c0c0",
        Block.WOOD: "#8b4513",
        Block.STONE: "#808080",
        Block.SAND: "#f4d03f",
    }

    def __init__(self, game):
        self.game = game

        self.root = tk.Tk()
        self.root.title("Blocks Game")

        canvas_size = game.size * self.CELL_SIZE+15

        self.canvas = tk.Canvas(
            self.root,
            width=canvas_size,
            height=canvas_size,
        )
        self.canvas.pack()

        self.info = tk.Label(self.root)
        self.info.pack()

        self.bind_keys()

        self.draw()

    def bind_keys(self):
        self.root.bind("<Left>", lambda e: self.action_move("left"))
        self.root.bind("<Right>", lambda e: self.action_move("right"))
        self.root.bind("<Up>", lambda e: self.action_move("up"))
        self.root.bind("<Down>", lambda e: self.action_move("down"))

        self.root.bind("w", lambda e: self.action_break("up"))
        self.root.bind("a", lambda e: self.action_break("left"))
        self.root.bind("s", lambda e: self.action_break("down"))
        self.root.bind("d", lambda e: self.action_break("right"))

        self.root.bind("<Shift-W>", lambda e: self.action_place("up"))
        self.root.bind("<Shift-A>", lambda e: self.action_place("left"))
        self.root.bind("<Shift-S>", lambda e: self.action_place("down"))
        self.root.bind("<Shift-D>", lambda e: self.action_place("right"))

        for i in range(1, 6):
            self.root.bind(
                str(i),
                lambda e, value=i: self.select(value)
            )

    def action_move(self, direction):
        self.game.move(direction)
        self.draw()

    def action_break(self, direction):
        self.game.break_block(direction)
        self.draw()

    def action_place(self, direction):
        self.game.place_block(direction)
        self.draw()

    def select(self, index):
        self.game.select_block(index)
        self.draw()

    def draw(self):
        self.canvas.delete("all")

        state = self.game.get_state()

        world = state["map"]

        for y in range(self.game.size):
            for x in range(self.game.size):

                block = Block(world[y, x])

                color = self.COLORS[block]

                x1 = x * self.CELL_SIZE
                y1 = y * self.CELL_SIZE
                x2 = x1 + self.CELL_SIZE
                y2 = y1 + self.CELL_SIZE

                self.canvas.create_rectangle(
                    x1, y1, x2, y2,
                    fill=color,
                    outline=""
                )

        px = state["player_x"]
        py = state["player_y"]

        self.canvas.create_oval(
            px * self.CELL_SIZE + 4,
            py * self.CELL_SIZE + 4,
            px * self.CELL_SIZE + self.CELL_SIZE - 4,
            py * self.CELL_SIZE + self.CELL_SIZE - 4,
            fill="red"
        )

        inv = state["inventory"]

        text = (
            f"Selected: {state['selected'].name} | "
            f"DIRT={inv[Block.DIRT]} "
            f"IRON={inv[Block.IRON]} "
            f"WOOD={inv[Block.WOOD]} "
            f"STONE={inv[Block.STONE]} "
            f"SAND={inv[Block.SAND]}"
        )

        self.info.config(text=text)

    def run(self):
        self.root.mainloop()
    @staticmethod
    def playback(game, gameplay, speed=1.0):
        gui = GameGUI(game)
        delay_ms = int(speed * 1000)

        index = 0

        def step():
            nonlocal index

            if index >= len(gameplay):
                return

            frame = gameplay[index]
            index += 1

            # FULL STATE RESTORE (important!)
            game.map = frame["map"].copy()
            game.player_x = frame["player_x"]
            game.player_y = frame["player_y"]
            game.inventory = frame["inventory"].copy()
            game.selected_block = frame["selected"]

            gui.draw()
            gui.root.after(delay_ms, step)

        gui.root.after(delay_ms, step)
        gui.run()

In [99]:
GameGUI(Game()).run()

In [107]:
def sample_gameplay(game: "Game", steps: int, valid_prob=0.8):
    all_actions = list(game.actions.keys())
    placeable = list(Block)[2:]
    directions = ["up", "down", "left", "right"]

    for _ in range(steps):

        if random.random() < valid_prob:
            actions = game.get_valid_actions()

            # split actions
            move_break = [a for a in actions if not a.startswith("place_")]
            place_actions = [a for a in actions if a.startswith("place_")]

            if place_actions and random.random() < 0.5:
                # =========================
                # PLACE LOGIC (2-stage)
                # =========================

                block_choices = [
                    b for b in placeable
                    if game.inventory[b] > 0
                    and game.selected_block != b
                ]

                if block_choices:
                    block = random.choice(block_choices)

                    # pick random direction
                    direction = random.choice(directions)

                    # simulate placement manually
                    game.selected_block = block
                    game.place_block(direction)

                    action_name = f"place_{block.name}_{direction}"
                else:
                    action_name = random.choice(move_break)

            else:
                action_name = random.choice(move_break)
                game.actions[action_name]()

        else:
            action_name = random.choice(all_actions)
            game.actions[action_name]()

        yield {
            "map": game.map.copy(),
            "player_x": game.player_x,
            "player_y": game.player_y,
            "inventory": game.inventory.copy(),
            "selected": game.selected_block,
            "action_taken": action_name
        }

In [108]:
game = Game()

gameplay = list(sample_gameplay(game, 1000,1))

GameGUI.playback(game, gameplay, speed=0.01)

In [97]:
gameplay[400]['selected']

<Block.SAND: 6>